# Bank Customer Churn Prediction — Imbalance Techniques

**Goal:** Predict whether a bank customer will churn (`Exited = 1`).

This notebook keeps the core workflow of the previous project, but adds a careful comparison of:

1. Baseline models without resampling
2. **SMOTE** synthetic oversampling
3. **Cost-sensitive learning / class weighting**
4. **Decision-threshold tuning**
5. **Anomaly detection as an exploratory diagnostic** — not treated as the primary churn model

The validation set remains untouched by SMOTE or preprocessing fitting. Outliers are detected and investigated, but plausible customer observations are not automatically deleted.

In [ ]:
# 1. IMPORT LIBRARIES

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (
    roc_auc_score, roc_curve, average_precision_score,
    accuracy_score, precision_score, recall_score, f1_score,
    log_loss, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, precision_recall_curve
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42

## 2. Load the data

Only `train.csv` is used because it contains the target. The competition `test.csv` is deliberately not used for model evaluation.

In [ ]:
train = pd.read_csv("train.csv")
print("Train shape:", train.shape)
display(train.head())

## 3. Basic data integrity checks

In [ ]:
print("Data types and missing values:")
train.info()
print("\nMissing values:")
display(train.isna().sum())

cols_without_id = train.columns.drop("id")
cols_without_id_target = train.columns.drop(["id", "Exited"])

print("\nDuplicate rows (ignoring id):", train.duplicated(subset=cols_without_id).sum())
print("Duplicate feature rows (ignoring id and target):", train.duplicated(subset=cols_without_id_target).sum())

n_rows = len(train)
print("\nUnique CustomerId:", train["CustomerId"].nunique(), f"({train['CustomerId'].nunique()/n_rows:.1%} of rows)")
print("Unique Surname:", train["Surname"].nunique(), f"({train['Surname'].nunique()/n_rows:.1%} of rows)")

## 4. Target distribution and class imbalance

`Exited = 1` is the churn class. The churn class is the minority class, so accuracy alone will not be sufficient for model comparison.

In [ ]:
target_counts = train["Exited"].value_counts().sort_index()
target_percent = train["Exited"].value_counts(normalize=True).sort_index() * 100

target_table = pd.DataFrame({"Count": target_counts, "Percentage": target_percent.round(2)})
display(target_table)

plt.figure(figsize=(6,4))
train["Exited"].value_counts().sort_index().plot(kind="bar")
plt.title("Customer Churn Distribution")
plt.xlabel("Exited (1 = Churned)")
plt.ylabel("Number of Customers")
plt.show()

## 5. EDA — categorical churn patterns

In [ ]:
categorical_eda = ["Geography", "Gender", "HasCrCard", "IsActiveMember"]

for col in categorical_eda:
    percentage_table = pd.crosstab(train[col], train["Exited"], normalize="index") * 100
    percentage_table.columns = ["Stayed (0)", "Churned (1)"]
    print(f"\n{col} — percentages within each group")
    display(percentage_table.round(2))

    plt.figure(figsize=(6,4))
    train.groupby(col)["Exited"].mean().plot(kind="bar")
    plt.title(f"Churn Rate by {col}")
    plt.ylabel("Churn Rate")
    plt.xlabel(col)
    plt.xticks(rotation=20)
    plt.show()

## 6. EDA — numerical variables, skewness and boxplots

Skewness is inspected rather than automatically transformed. A skewed predictor is not automatically a bad predictor. We will test transformations only if they improve validation performance.

In [ ]:
numerical_features = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary"
]

print("Numerical summary:")
display(train[numerical_features].describe().T)

# Continuous variables for skewness — binary indicators are excluded.
continuous_features = [
    "CreditScore", "Age", "Tenure", "Balance", "NumOfProducts", "EstimatedSalary"
]

skewness = train[continuous_features].skew().sort_values(ascending=False)
print("Skewness of continuous numerical variables:")
display(skewness.round(3))

for col in continuous_features:
    plt.figure(figsize=(7,4))
    plt.hist(train[col], bins=30)
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.title(f"Distribution of {col}")
    plt.show()

for col in numerical_features:
    plt.figure(figsize=(7,4))
    train.boxplot(column=col, by="Exited")
    plt.title(f"{col} vs Churn")
    plt.show()

### Interpretation of skewness

The earlier analysis showed that **Age is the most noticeably right-skewed variable**, while Balance and NumOfProducts are only mildly right-skewed and the remaining continuous variables are closer to symmetric. We therefore do not blindly log-transform every variable.

## 7. IQR-based outlier detection

The IQR rule is used as a **diagnostic**, not as an automatic deletion rule.

A statistical outlier can still be a genuine bank customer. For example, older customers or customers with an unusual number of products may be valid observations. Removing them without evidence of data error can throw away useful churn signal.

In [ ]:
outlier_summary = []

for col in continuous_features:
    Q1 = train[col].quantile(0.25)
    Q3 = train[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    mask = (train[col] < lower) | (train[col] > upper)

    outlier_summary.append({
        "Variable": col,
        "Q1": Q1,
        "Q3": Q3,
        "Lower Bound": lower,
        "Upper Bound": upper,
        "Outliers": int(mask.sum()),
        "Percentage": mask.mean() * 100
    })

outlier_summary = pd.DataFrame(outlier_summary)
display(outlier_summary.round(2))

### Outlier decision

**Decision: retain the observations.**

The IQR rule flags observations that are unusual relative to the middle 50% of the data; it does not prove that they are errors. We therefore do not delete or winsorize these observations. The models receive the original customer data, while the tree models can naturally handle non-linear effects and the Logistic Regression pipeline uses scaling.

## 8. Correlation and feature selection

In [ ]:
plt.figure(figsize=(10,7))
sns.heatmap(train.select_dtypes(include=np.number).corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

DROP_COLUMNS = ["id", "CustomerId", "Surname"]
X = train.drop(columns=["Exited"] + DROP_COLUMNS)
y = train["Exited"]

print("Features after removing identifiers:", X.shape)
display(X.head())

## 9. Stratified train-validation split

We use an 80/20 split with `stratify=y`. This **does not remove class imbalance**; it preserves approximately the same class proportions in both sets.

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("X_train:", X_train.shape)
print("X_valid:", X_valid.shape)
print("\nChurn rate:")
print("Train     :", round(y_train.mean(), 4))
print("Validation:", round(y_valid.mean(), 4))

## 10. Leakage-safe preprocessing

Categorical variables are one-hot encoded. Numerical variables are median-imputed and standardized for Logistic Regression.

**Important:** preprocessing is inside the pipeline. Therefore it is fitted using the training data only.

SMOTE is also placed inside an `imblearn` pipeline so synthetic samples are generated **only during model fitting**, never on the validation set.

In [ ]:
categorical_features = ["Geography", "Gender"]

numeric_transformer_scaled = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

preprocessor_scaled = ColumnTransformer([
    ("num", numeric_transformer_scaled, numerical_features),
    ("cat", categorical_transformer, categorical_features)
])

tree_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), numerical_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", drop="first"))
    ]), categorical_features)
])

## 11. Baseline models — no imbalance intervention

These models establish the reference performance before SMOTE or cost-sensitive learning.

In [ ]:
logistic_model = Pipeline([
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

tree_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=20, random_state=RANDOM_STATE
    ))
])

rf_model = Pipeline([
    ("preprocessor", tree_preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100, max_depth=10, min_samples_leaf=10,
        n_jobs=-1, random_state=RANDOM_STATE
    ))
])

baseline_models = {
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": rf_model
}

baseline_predictions = {}

for name, model in baseline_models.items():
    model.fit(X_train, y_train)
    baseline_predictions[name] = model.predict_proba(X_valid)[:, 1]

print("Baseline models fitted successfully.")

## 12. SMOTE models

SMOTE creates synthetic minority-class observations by interpolating between minority examples. It is **not** simple duplication.

The critical rule is: **SMOTE is applied only to the training portion inside the pipeline.** The validation set remains in its original distribution.

In [ ]:
smote_models = {
    "Logistic Regression + SMOTE": ImbPipeline([
        ("preprocessor", preprocessor_scaled),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "Decision Tree + SMOTE": ImbPipeline([
        ("preprocessor", tree_preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", DecisionTreeClassifier(
            max_depth=6, min_samples_leaf=20, random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest + SMOTE": ImbPipeline([
        ("preprocessor", tree_preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("model", RandomForestClassifier(
            n_estimators=100, max_depth=10, min_samples_leaf=10,
            n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}

smote_predictions = {}
for name, model in smote_models.items():
    model.fit(X_train, y_train)
    smote_predictions[name] = model.predict_proba(X_valid)[:, 1]

print("SMOTE models fitted successfully.")

## 13. Cost-sensitive models — class weighting

Instead of changing the data, class weighting changes the penalty assigned to errors from each class. `class_weight="balanced"` automatically gives more weight to the minority class.

In [ ]:
weighted_models = {
    "Logistic Regression + Class Weight": Pipeline([
        ("preprocessor", preprocessor_scaled),
        ("model", LogisticRegression(
            class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE
        ))
    ]),
    "Decision Tree + Class Weight": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", DecisionTreeClassifier(
            class_weight="balanced", max_depth=6, min_samples_leaf=20,
            random_state=RANDOM_STATE
        ))
    ]),
    "Random Forest + Class Weight": Pipeline([
        ("preprocessor", tree_preprocessor),
        ("model", RandomForestClassifier(
            class_weight="balanced", n_estimators=100, max_depth=10,
            min_samples_leaf=10, n_jobs=-1, random_state=RANDOM_STATE
        ))
    ])
}

weighted_predictions = {}
for name, model in weighted_models.items():
    model.fit(X_train, y_train)
    weighted_predictions[name] = model.predict_proba(X_valid)[:, 1]

print("Class-weighted models fitted successfully.")

## 14. Compare baseline, SMOTE and class weighting

The validation set is identical for every experiment. We compare probability-based metrics (ROC-AUC, PR-AUC, Log Loss) and threshold-based metrics at the default 0.50 threshold.

In [ ]:
def evaluate_model(y_true, probabilities, threshold=0.50):
    predictions = (probabilities >= threshold).astype(int)
    return {
        "ROC-AUC": roc_auc_score(y_true, probabilities),
        "PR-AUC": average_precision_score(y_true, probabilities),
        "Log Loss": log_loss(y_true, probabilities),
        "Accuracy": accuracy_score(y_true, predictions),
        "Precision": precision_score(y_true, predictions, zero_division=0),
        "Recall": recall_score(y_true, predictions, zero_division=0),
        "F1-Score": f1_score(y_true, predictions, zero_division=0)
    }

all_predictions = {}
all_predictions.update(baseline_predictions)
all_predictions.update(smote_predictions)
all_predictions.update(weighted_predictions)

evaluation_results = pd.DataFrame({
    name: evaluate_model(y_valid, pred)
    for name, pred in all_predictions.items()
}).T

display(evaluation_results.sort_values("PR-AUC", ascending=False).round(4))

## 15. Direct comparison of each imbalance technique against its baseline

This table makes the decision transparent: positive differences mean the technique improved the metric.

In [ ]:
comparison_rows = []
for base_name, base_pred in baseline_predictions.items():
    short = base_name
    smote_name = f"{base_name} + SMOTE"
    weight_name = f"{base_name} + Class Weight"

    base_metrics = evaluate_model(y_valid, base_pred)
    smote_metrics = evaluate_model(y_valid, smote_predictions[smote_name])
    weight_metrics = evaluate_model(y_valid, weighted_predictions[weight_name])

    for technique, metrics in [("Baseline", base_metrics), ("SMOTE", smote_metrics), ("Class Weight", weight_metrics)]:
        comparison_rows.append({"Base Model": base_name, "Technique": technique, **metrics})

technique_comparison = pd.DataFrame(comparison_rows)
display(technique_comparison.sort_values(["Base Model", "PR-AUC"], ascending=[True, False]).round(4))

## 16. Train vs validation ROC-AUC — overfitting check

In [ ]:
overfit_rows = []
for name, model in baseline_models.items():
    train_auc = roc_auc_score(y_train, model.predict_proba(X_train)[:, 1])
    valid_auc = roc_auc_score(y_valid, baseline_predictions[name])
    overfit_rows.append({
        "Model": name,
        "Train ROC-AUC": train_auc,
        "Validation ROC-AUC": valid_auc,
        "Gap": train_auc - valid_auc
    })

display(pd.DataFrame(overfit_rows).set_index("Model").round(4))

## 17. ROC and Precision-Recall curves

In [ ]:
plt.figure(figsize=(9,6))
for name, pred in all_predictions.items():
    fpr, tpr, _ = roc_curve(y_valid, pred)
    auc = roc_auc_score(y_valid, pred)
    plt.plot(fpr, tpr, label=f"{name} (AUC={auc:.3f})")
plt.plot([0,1], [0,1], linestyle="--", label="Random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves")
plt.legend(fontsize=8)
plt.show()

plt.figure(figsize=(9,6))
for name, pred in all_predictions.items():
    precision, recall, _ = precision_recall_curve(y_valid, pred)
    ap = average_precision_score(y_valid, pred)
    plt.plot(recall, precision, label=f"{name} (AP={ap:.3f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curves")
plt.legend(fontsize=8)
plt.show()

## 18. Threshold tuning — business decision layer

ROC-AUC and PR-AUC are based on probability rankings and do not depend on one fixed classification threshold. Precision, recall and F1 do.

We therefore examine thresholds on the validation predictions of the strongest candidate. **This is threshold analysis, not a claim that the 0.50 threshold is optimal.** In a production setting, the final threshold should be selected using a separate validation/calibration set or cross-validation and then evaluated on a truly untouched test set.

In [ ]:
# Select the best probability-ranking candidate by validation PR-AUC.
best_name = evaluation_results["PR-AUC"].idxmax()
best_pred = all_predictions[best_name]

threshold_rows = []
for threshold in np.arange(0.10, 0.91, 0.01):
    pred_class = (best_pred >= threshold).astype(int)
    threshold_rows.append({
        "Threshold": threshold,
        "Precision": precision_score(y_valid, pred_class, zero_division=0),
        "Recall": recall_score(y_valid, pred_class, zero_division=0),
        "F1-Score": f1_score(y_valid, pred_class, zero_division=0)
    })

threshold_results = pd.DataFrame(threshold_rows)
best_f1_row = threshold_results.loc[threshold_results["F1-Score"].idxmax()]

print("Best probability-ranking candidate:", best_name)
print("Threshold with maximum validation F1:")
display(best_f1_row.to_frame().T.round(4))

display(threshold_results.iloc[::5].round(3))

plt.figure(figsize=(9,5))
plt.plot(threshold_results["Threshold"], threshold_results["Precision"], label="Precision")
plt.plot(threshold_results["Threshold"], threshold_results["Recall"], label="Recall")
plt.plot(threshold_results["Threshold"], threshold_results["F1-Score"], label="F1")
plt.axvline(0.50, linestyle="--", label="Default 0.50")
plt.xlabel("Classification Threshold")
plt.ylabel("Score")
plt.title(f"Threshold Analysis — {best_name}")
plt.legend()
plt.show()

## 19. Confusion matrices at 0.50 and tuned threshold

In [ ]:
tuned_threshold = float(best_f1_row["Threshold"])

for threshold in [0.50, tuned_threshold]:
    pred_class = (best_pred >= threshold).astype(int)
    print(f"\nThreshold = {threshold:.2f}")
    print(classification_report(y_valid, pred_class, target_names=["Stayed", "Churned"], zero_division=0))

    cm = confusion_matrix(y_valid, pred_class)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Stayed", "Churned"])
    disp.plot()
    plt.title(f"Confusion Matrix — {best_name}, threshold={threshold:.2f}")
    plt.show()

## 20. Anomaly detection — exploratory diagnostic, not the main churn solution

Anomaly detection answers a different question: **which customers look unusual?** It does not directly model the labelled probability of churn.

We therefore use Isolation Forest only as an exploratory comparison. A high anomaly score is not automatically equivalent to churn. If its churn ROC-AUC is weak, that is evidence that unusualness is not a good substitute for supervised churn modelling.

In [ ]:
# Transform the feature space using the same leakage-safe tree preprocessing.
X_train_transformed = tree_preprocessor.fit_transform(X_train)
X_valid_transformed = tree_preprocessor.transform(X_valid)

iso_forest = IsolationForest(
    n_estimators=100,
    contamination="auto",
    random_state=RANDOM_STATE,
    n_jobs=-1
)
iso_forest.fit(X_train_transformed)

# IsolationForest: larger decision_function = more normal.
# Negating it gives an anomaly score where larger = more anomalous.
anomaly_score = -iso_forest.decision_function(X_valid_transformed)

print("Isolation Forest anomaly score ROC-AUC for churn:", round(roc_auc_score(y_valid, anomaly_score), 4))
print("Isolation Forest anomaly score PR-AUC for churn:", round(average_precision_score(y_valid, anomaly_score), 4))

### Why anomaly detection is not selected as the churn model

The target is labelled (`Exited`), so this is fundamentally a **supervised classification problem**. Isolation Forest can identify unusual customers, but an unusual customer is not necessarily a customer who will churn. Therefore anomaly detection is kept as an exploratory diagnostic rather than replacing the supervised models.

## 21. Final decision framework

Use the results above rather than assuming one technique must win.

- If **SMOTE** improves PR-AUC/F1/recall without an unacceptable precision or log-loss trade-off, it can be justified.
- If **class weighting** performs better, it is attractive because it keeps the original observations and changes the error cost rather than synthesizing observations.
- If ranking metrics are similar but the business needs more churners detected, **threshold tuning** can adjust recall/precision without retraining the model.
- **Anomaly detection** is not treated as a replacement for supervised churn prediction.
- Outliers are retained because IQR flags statistical extremes, not necessarily erroneous records.
- Skewness is documented, but transformations are not applied merely because a variable is skewed; predictive validation performance should justify the transformation.

## 22. Interview-ready conclusion

> “I first established baseline Logistic Regression, Decision Tree and Random Forest models. Because the churn class was the minority class, I evaluated ROC-AUC and PR-AUC along with precision, recall and F1 rather than relying on accuracy. I then compared SMOTE and cost-sensitive class weighting on the same untouched validation set. I also examined threshold tuning because changing the decision threshold can trade precision for recall without changing the underlying probability ranking. I considered anomaly detection, but it answers an unsupervised ‘unusual customer’ question rather than directly predicting the labelled churn outcome, so I kept it as a diagnostic rather than the final approach. I retained statistically unusual but plausible customer observations rather than deleting them automatically.”